RunnableParallel

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [4]:
chat_template_books = ChatPromptTemplate.from_template(
    """
    Suggest three of the best intermediate-level {programming_language} books.
    Answer only by listing the books.
    """
)

chat_template_projects = ChatPromptTemplate.from_template(
    """
    Suggest three interesting {programming_language} projects suitable for intermediate-level programmers.
    Answer only by listing the projects.
    """
)

chat_template_time = ChatPromptTemplate.from_template(
    """
    I'm an intermediate level programmer.

    Consider the following literature:
    {books}

    Also, consider the following projects:
    {projects}

    Roughly how much time would it take me to complete the literature and the projects?
    """
)


In [5]:
chat = ChatGroq(model_name="llama-3.1-8b-instant",
                model_kwargs= {'seed':365},
                temperature = 0,
                max_tokens= 500)

In [6]:
string_parser = StrOutputParser()

In [7]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [8]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [9]:
chain_parallel.invoke({'programming_language':'Python'})

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame or Pyglet'}

In [ ]:
chain_time1 = (RunnableParallel({'books':chain_books,
                                 'projects':chain_projects}) | chat_template_time
                                 | chat
                                 | string_parser)

In [17]:
chain_time2 = ({'books':chain_books,
                                 'projects':chain_projects} | chat_template_time
                                 | chat
                                 | string_parser)

In [18]:
print(chain_time2.invoke({'programming_language':'python'}))

To estimate the time required to complete the literature and projects, let's break down the tasks into smaller components.

**Literature:**

1. "Automate the Boring Stuff with Python" by Al Sweigart:
   - Estimated reading time: 10-15 hours (depending on your reading speed and pace)
   - This book focuses on practical applications of Python, so it's likely you'll spend some time practicing the code examples.

2. "Python Crash Course" by Eric Matthes:
   - Estimated reading time: 20-25 hours (depending on your reading speed and pace)
   - This book covers a wide range of topics, including data structures, file input/output, and web development. You may need to spend more time practicing and experimenting with the code examples.

3. "Learning Python" by Mark Lutz:
   - Estimated reading time: 30-40 hours (depending on your reading speed and pace)
   - This book is a comprehensive guide to the Python language, covering topics from basic syntax to advanced concepts. It's likely to take mor

In [19]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  